# Etapa 1 — Preprocesamiento y Data Leakage
## Predicción de Falla Cardíaca

**Objetivo de este notebook:**
1. Cargar y explorar el dataset `heart.csv`.
2. Demostrar el efecto del *data leakage* comparando un flujo incorrecto (escalar antes de dividir) contra uno correcto (Pipeline).
3. Entrenar y comparar 5 modelos distintos usando `Pipeline` + `GridSearchCV`, evaluando con AUC y Accuracy.
4. Construir un ranking final de modelos.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix, ConfusionMatrixDisplay, roc_curve

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 1. Carga y exploración de datos

Ajusta la ruta si guardaste `heart.csv` en otra carpeta.

In [9]:
df = pd.read_csv("C:\\Users\\Katherin Barrera\\Downloads\\heart-disease-mlops\\heart-disease-mlops\\notebooks\\heart.csv")
print(df.shape)
df.head()

(918, 12)


,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 918 entries, 0 to 917
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Age             918 non-null    int64  
 1   Sex             918 non-null    object 
 2   ChestPainType   918 non-null    object 
 3   RestingBP       918 non-null    int64  
 4   Cholesterol     918 non-null    int64  
 5   FastingBS       918 non-null    int64  
 6   RestingECG      918 non-null    object 
 7   MaxHR           918 non-null    int64  
 8   ExerciseAngina  918 non-null    object 
 9   Oldpeak         918 non-null    float64
 10  ST_Slope        918 non-null    object 
 11  HeartDisease    918 non-null    int64  
dtypes: float64(1), int64(6), object(5)
memory usage: 86.2+ KB


In [11]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Age,918.0,NaN,NaN,NaN,53.510893,9.432617,28.0,47.0,54.0,60.0,77.0
Sex,918,2,M,725,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ChestPainType,918,4,ASY,496,NaN,NaN,NaN,NaN,NaN,NaN,NaN
RestingBP,918.0,NaN,NaN,NaN,132.396514,18.514154,0.0,120.0,130.0,140.0,200.0
Cholesterol,918.0,NaN,NaN,NaN,198.799564,109.384145,0.0,173.25,223.0,267.0,603.0
FastingBS,918.0,NaN,NaN,NaN,0.233115,0.423046,0.0,0.0,0.0,0.0,1.0
RestingECG,918,3,Normal,552,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MaxHR,918.0,NaN,NaN,NaN,136.809368,25.460334,60.0,120.0,138.0,156.0,202.0
ExerciseAngina,918,2,N,547,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Oldpeak,918.0,NaN,NaN,NaN,0.887364,1.06657,-2.6,0.0,0.6,1.5,6.2


In [12]:
# Distribución de la variable objetivo
target_col = "HeartDisease"  # ajusta el nombre si tu dataset usa otro
df[target_col].value_counts(normalize=True)

HeartDisease
1    0.553377
0    0.446623
Name: proportion, dtype: float64

In [20]:
import plotly.express as px

fig = px.histogram(
    df,
    x=target_col,
    color=target_col,
    title="Distribución de la variable objetivo",
    text_auto=True
)
fig.update_layout(bargap=0.2, showlegend=False)
fig.show()

## 2. Separación de features y target

Identificamos columnas numéricas y categóricas para tratarlas de forma distinta dentro del pipeline.

In [14]:
X = df.drop(columns=[target_col])
y = df[target_col]

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Numéricas:", numeric_features)
print("Categóricas:", categorical_features)

Numéricas: ['Age', 'RestingBP', 'Cholesterol', 'FastingBS', 'MaxHR', 'Oldpeak']
Categóricas: ['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope']


## 3. Demostración de Data Leakage

**Flujo INCORRECTO (con fuga de datos):** se escala `X` completo *antes* de dividir en train/test.
El escalador "ve" estadísticas (media, desviación) de los datos de test durante el ajuste, lo cual infla artificialmente el rendimiento.

**Flujo CORRECTO:** se divide primero, y el escalador se ajusta *solo* con los datos de entrenamiento (dentro de un `Pipeline`).

In [15]:
# --- FLUJO INCORRECTO: escalar antes de dividir ---
scaler_leak = StandardScaler()
X_numeric = X[numeric_features]
X_scaled_leak = scaler_leak.fit_transform(X_numeric)  # fit sobre TODO el dataset -> fuga

X_train_leak, X_test_leak, y_train_leak, y_test_leak = train_test_split(
    X_scaled_leak, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

model_leak = SVC(probability=True, random_state=RANDOM_STATE)
model_leak.fit(X_train_leak, y_train_leak)
y_pred_leak = model_leak.predict(X_test_leak)
y_proba_leak = model_leak.predict_proba(X_test_leak)[:, 1]

print("=== Flujo CON leakage ===")
print("Accuracy:", accuracy_score(y_test_leak, y_pred_leak))
print("AUC:", roc_auc_score(y_test_leak, y_proba_leak))

=== Flujo CON leakage ===
Accuracy: 0.8206521739130435
AUC: 0.8877331420373027


In [17]:
# --- FLUJO CORRECTO: dividir primero, luego pipeline con escalado ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

pipeline_correct = Pipeline([
    ("scaler", StandardScaler()),
    ("model", SVC(probability=True, random_state=RANDOM_STATE))
])

# El scaler se ajusta SOLO con X_train dentro del pipeline
pipeline_correct.fit(X_train[numeric_features], y_train)
y_pred_correct = pipeline_correct.predict(X_test[numeric_features])
y_proba_correct = pipeline_correct.predict_proba(X_test[numeric_features])[:, 1]

print("=== Flujo CORRECTO (sin leakage) ===")
print("Accuracy:", accuracy_score(y_test, y_pred_correct))
print("AUC:", roc_auc_score(y_test, y_proba_correct))

=== Flujo CORRECTO (sin leakage) ===
Accuracy: 0.8206521739130435
AUC: 0.888271162123386


En datasets pequeños la diferencia numérica entre ambos flujos puede ser sutil, pero el punto conceptual es crítico: el flujo con leakage usa información de test durante el entrenamiento, algo que en producción nunca tendrías disponible. Por eso, de aquí en adelante trabajamos **siempre** con `Pipeline` + partición previa.

## 4. Preprocesamiento completo con `ColumnTransformer`

Construimos un preprocesador que:
- Escala las variables numéricas (`StandardScaler`).
- Codifica las variables categóricas (`OneHotEncoder`).

In [21]:
preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])

## 5. Función reutilizable de entrenamiento

`train_pipeline` arma un `Pipeline` (preprocesamiento + modelo), corre `GridSearchCV` con la grilla de hiperparámetros dada, y devuelve el mejor estimador junto con sus métricas en test.

In [ ]:
def train_pipeline(model, param_grid, X_train, y_train, X_test, y_test, cv=5, scoring="roc_auc"):
    """Entrena un modelo dentro de un Pipeline con GridSearchCV y evalúa en test.

    Parameters
    ----------
    model : estimador de sklearn (sin ajustar)
    param_grid : dict con hiperparámetros para GridSearchCV (prefijo 'model__')
    X_train, y_train, X_test, y_test : splits de datos
    cv : número de folds
    scoring : métrica para seleccionar el mejor modelo

    Returns
    -------
    dict con el pipeline ajustado, mejores parámetros, y métricas de test
    """
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    grid = GridSearchCV(pipe, param_grid=param_grid, cv=cv, scoring=scoring, n_jobs=-1)
    grid.fit(X_train, y_train)

    best_pipe = grid.best_estimator_
    y_pred = best_pipe.predict(X_test)
    y_proba = best_pipe.predict_proba(X_test)[:, 1]

    return {
        "name": model.__class__.__name__,
        "best_estimator": best_pipe,
        "best_params": grid.best_params_,
        "cv_best_score": grid.best_score_,
        "test_accuracy": accuracy_score(y_test, y_pred),
        "test_auc": roc_auc_score(y_test, y_proba),
        "y_pred": y_pred,
        "y_proba": y_proba,
    }

## 6. Entrenamiento y comparación de 5 modelos

Entrenamos: **Logistic Regression, Random Forest, KNN, Gradient Boosting y SVC**, cada uno con su propia grilla de hiperparámetros.

In [23]:
model_configs = [
    (LogisticRegression(max_iter=1000, random_state=RANDOM_STATE), {
        "model__C": [0.01, 0.1, 1, 10],
        "model__penalty": ["l2"],
    }),
    (RandomForestClassifier(random_state=RANDOM_STATE), {
        "model__n_estimators": [100, 200],
        "model__max_depth": [None, 5, 10],
    }),
    (KNeighborsClassifier(), {
        "model__n_neighbors": [3, 5, 7, 9],
        "model__weights": ["uniform", "distance"],
    }),
    (GradientBoostingClassifier(random_state=RANDOM_STATE), {
        "model__n_estimators": [100, 200],
        "model__learning_rate": [0.05, 0.1],
        "model__max_depth": [2, 3],
    }),
    (SVC(probability=True, random_state=RANDOM_STATE), {
        "model__C": [0.1, 1, 10],
        "model__kernel": ["rbf", "linear"],
    }),
]

results = []
for model, grid_params in model_configs:
    print(f"Entrenando {model.__class__.__name__}...")
    res = train_pipeline(model, grid_params, X_train, y_train, X_test, y_test)
    results.append(res)
    print(f"  -> AUC test: {res['test_auc']:.4f} | Accuracy test: {res['test_accuracy']:.4f}")
    print(f"  -> Mejores params: {res['best_params']}\n")

Entrenando LogisticRegression...
  -> AUC test: 0.9323 | Accuracy test: 0.8913
  -> Mejores params: {'model__C': 0.1, 'model__penalty': 'l2'}

Entrenando RandomForestClassifier...
  -> AUC test: 0.9362 | Accuracy test: 0.8967
  -> Mejores params: {'model__max_depth': 5, 'model__n_estimators': 100}

Entrenando KNeighborsClassifier...
  -> AUC test: 0.9534 | Accuracy test: 0.9239
  -> Mejores params: {'model__n_neighbors': 9, 'model__weights': 'distance'}

Entrenando GradientBoostingClassifier...
  -> AUC test: 0.9310 | Accuracy test: 0.9022
  -> Mejores params: {'model__learning_rate': 0.05, 'model__max_depth': 2, 'model__n_estimators': 100}

Entrenando SVC...
  -> AUC test: 0.9494 | Accuracy test: 0.8967
  -> Mejores params: {'model__C': 1, 'model__kernel': 'rbf'}



## 7. Ranking de modelos

In [24]:
ranking = pd.DataFrame([
    {
        "Modelo": r["name"],
        "AUC (test)": r["test_auc"],
        "Accuracy (test)": r["test_accuracy"],
        "CV Best Score": r["cv_best_score"],
    }
    for r in results
]).sort_values("AUC (test)", ascending=False).reset_index(drop=True)

ranking

,Modelo,AUC (test),Accuracy (test),CV Best Score
0,KNeighborsClassifier,0.953431,0.923913,0.914103
1,SVC,0.949426,0.896739,0.924232
2,RandomForestClassifier,0.936155,0.896739,0.928768
3,LogisticRegression,0.932329,0.891304,0.922562
4,GradientBoostingClassifier,0.930954,0.902174,0.924183


In [26]:
import plotly.express as px

fig = px.bar(
    ranking.sort_values("AUC (test)", ascending=True),  # ascendente para que el mejor quede arriba en horizontal
    x="AUC (test)",
    y="Modelo",
    orientation="h",
    color="AUC (test)",
    color_continuous_scale="viridis",
    text="AUC (test)",
    title="Ranking de modelos por AUC en test"
)

fig.update_traces(texttemplate="%{text:.3f}", textposition="outside")
fig.update_layout(xaxis_range=[0, 1], coloraxis_showscale=False)
fig.show()

## 8. Curvas ROC comparadas

In [28]:
import plotly.graph_objects as go

fig = go.Figure()

for r in results:
    fpr, tpr, _ = roc_curve(y_test, r["y_proba"])
    fig.add_trace(go.Scatter(
        x=fpr, y=tpr,
        mode="lines",
        name=f"{r['name']} (AUC={r['test_auc']:.3f})",
        hovertemplate="FPR: %{x:.3f}<br>TPR: %{y:.3f}<extra></extra>"
    ))

# Línea de referencia (azar)
fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1],
    mode="lines",
    name="Azar",
    line=dict(dash="dash", color="gray")
))

fig.update_layout(
    title="Curvas ROC — comparación de modelos",
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate",
    legend_title="Modelo",
    width=750,
    height=600
)
fig.show()

## 9. Matriz de confusión del mejor modelo

In [31]:
import plotly.graph_objects as go

best_result = max(results, key=lambda r: r["test_auc"])
print(f"Mejor modelo: {best_result['name']} (AUC={best_result['test_auc']:.4f})")

cm = confusion_matrix(y_test, best_result["y_pred"])
labels = ["No enfermedad", "Enfermedad"]  # ajusta según tus clases reales

fig = go.Figure(data=go.Heatmap(
    z=cm,
    x=labels,
    y=labels,
    colorscale="Blues",
    showscale=True,
    hovertemplate="Real: %{y}<br>Predicho: %{x}<br>Conteo: %{z}<extra></extra>"
))

# Anotaciones manuales con el conteo en cada celda
annotations = []
for i, row_label in enumerate(labels):
    for j, col_label in enumerate(labels):
        annotations.append(dict(
            x=col_label, y=row_label,
            text=str(cm[i, j]),
            showarrow=False,
            font=dict(color="white" if cm[i, j] > cm.max() / 2 else "black", size=16)
        ))

fig.update_layout(
    title=f"Matriz de confusión — {best_result['name']}",
    xaxis_title="Predicción",
    yaxis_title="Valor real",
    annotations=annotations,
    width=500,
    height=500
)
fig.update_yaxes(autorange="reversed")
fig.show()

Mejor modelo: KNeighborsClassifier (AUC=0.9534)


## Interpretacion

- El flujo con `Pipeline` evita la fuga de datos al garantizar que el preprocesamiento se ajusta únicamente con los datos de entrenamiento.
- De los 5 modelos evaluados, **`{{MEJOR_MODELO}}`** obtuvo el mejor desempeño según AUC en el conjunto de test.
- El mejor `best_estimator_` de este modelo se exportará en la Etapa 3 con `joblib.dump()` para su uso en la API de predicción.

> Reemplaza `{{MEJOR_MODELO}}` por el nombre real una vez ejecutes el notebook con tus datos.